In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
from google.colab import files
uploaded = files.upload()  # manually select images.zip
# # i used this untuk masukin dr my laptop to here, yg bawah cuman bisa kalo dr drive sendiri

Saving images.zip to images.zip
Saving dataset.csv to dataset.csv


In [ ]:
# ── Standard Library ──────────────────────────────────────────────────────────
import gc
import os
import re
import zipfile

# ── Data ──────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score
)

# ── PyTorch ───────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.amp import autocast, GradScaler

# ── Vision ────────────────────────────────────────────────────────────────────
import timm
from torchvision import transforms, models
from PIL import Image

# ── Transformers ──────────────────────────────────────────────────────────────
from transformers import (
    AutoTokenizer, AutoModel,
    XLMRobertaTokenizer, XLMRobertaModel,
    CLIPProcessor, CLIPModel,
    get_linear_schedule_with_warmup
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

Using device: cuda


In [ ]:
#Helper Function
def print_results(model_name, acc, prec, rec, f1, auc):
    print(f"\n── {model_name} Results ──")
    print(f"  Accuracy  : {acc:.4f}")
    print(f"  Precision : {prec:.4f}")
    print(f"  Recall    : {rec:.4f}")
    print(f"  F1-Score  : {f1:.4f}")
    print(f"  AUC-ROC   : {auc:.4f}")

def run_late_fusion(name, probs1, probs2, true_labels):
    fused_probs = (np.array(probs1) + np.array(probs2)) / 2.0
    fused_preds = (fused_probs >= 0.5).astype(int)
    acc       = accuracy_score(true_labels, fused_preds)
    precision = precision_score(true_labels, fused_preds, zero_division=0)
    recall    = recall_score(true_labels, fused_preds, zero_division=0)
    f1        = f1_score(true_labels, fused_preds, zero_division=0)
    auc       = roc_auc_score(true_labels, fused_probs)
    print_results(f"Late Fusion: {name}", acc, precision, recall, f1, auc)
    return fused_probs, fused_preds

In [ ]:
# ── Load & Prepare Data ───────────────────────────────────────────────────────

# load dataset
with zipfile.ZipFile("/content/images.zip", 'r') as zip_ref:
    zip_ref.extractall("/content/")

df = pd.read_csv("/content/HumoraiNewsDataset.csv")
df["label_encoded"] = df["Label"].map({"AI": 1, "Human": 0})
df = df.drop(columns=["News Source", "Article Link", "Label"])

# verify contents
print(os.listdir("/content"))
print(os.listdir("/content/images")[:5])

# full image paths
df["full_image_path"] = df["Image"].apply(lambda x: "/content/" + str(x))

# text cleaning
def clean_text(text):
    text = str(text)
    text = re.sub(r"<[^>]+>", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df["clean_text"] = df["Text (Include title)"].apply(clean_text)

df.head()

['.config', 'images.zip', 'indobert_test_probs.npy', 'xlmr_test_probs.npy', 'images', 'images (1).zip', 'y_test.npy', 'dataset.csv', 'dataset (1).csv', 'sample_data']
['A669.png', 'A323.jpeg', 'A064.jpg', 'A704.jpg', 'A552.jpg']


,ArticleID,Text (Include title),Image,Category,lable_encoded,full_image_path,clean_text
0,A001,Bridgestone Tawarkan Cek Kondisi Ban Mobil Sel...,images/A001.jpeg,Otomotif,0,/content/images/A001.jpeg,Bridgestone Tawarkan Cek Kondisi Ban Mobil Sel...
1,A002,Mudik Aman Lebaran 2026: Bridgestone Buka Laya...,images/A002.png,Otomotif,1,/content/images/A002.png,Mudik Aman Lebaran 2026: Bridgestone Buka Laya...
2,A003,Indomobil Emotor Belum Berminat Ikut Tren Sewa...,images/A003.jpeg,Otomotif,0,/content/images/A003.jpeg,Indomobil Emotor Belum Berminat Ikut Tren Sewa...
3,A004,"Indomobil Emotor Pilih Jalur Sendiri, Belum Te...",images/A004.png,Otomotif,1,/content/images/A004.png,"Indomobil Emotor Pilih Jalur Sendiri, Belum Te..."
4,A005,Auto2000 Sudah Kirim 900 Unit Toyota Veloz Hyb...,images/A005.png,Otomotif,0,/content/images/A005.png,Auto2000 Sudah Kirim 900 Unit Toyota Veloz Hyb...


In [ ]:
# Data splitting
X= df.drop("label_encoded",axis=1)
y= df["label_encoded"]
X_train, X_testNval, y_train, y_testNval = train_test_split(X,y,
                                                            test_size=.30,
                                                            random_state=42,
                                                            stratify=X["Category"])
X_val, X_test, y_val, y_test = train_test_split(X_testNval,y_testNval,
                                                test_size=.50,
                                                random_state=42,
                                                stratify=X_testNval["Category"])

print("Train size:", len(X_train))
print("Validation size:", len(X_val))
print("Test size:", len(X_test))

#X variables for text models
Xt_Train= X_train[["clean_text"]]
Xt_Val= X_val[["clean_text"]]
Xt_Test= X_test[["clean_text"]]
print(Xt_Train.head())

# X variables for image models
Xi_Train= X_train[["full_image_path"]]
Xi_Val=X_val[["full_image_path"]]
Xi_Test=X_test[["full_image_path"]]
print(Xi_Train.head())

Train size: 525
Validation size: 112
Test size: 113
                                            clean_text
81   Elon Musk Sindir Industri Otomotif, Pabrikan K...
535  Dialog Meja Bundar Hambalang: Presiden Prabowo...
234  Perkuat Ekonomi RI, Prabowo Usung Strategi 'In...
283  Bangun Ikatan Keluarga Kuat: Main dan Tertawa ...
541  Eskalasi Konflik Timur Tengah: Fasilitas Gas S...
               full_image_path
81    /content/images/A082.jpg
535   /content/images/A536.jpg
234  /content/images/A235.jpeg
283   /content/images/A284.png
541   /content/images/A542.jpg


In [ ]:
# text model (RemBERT)
if 'model' in dir():
    del model
gc.collect()
torch.cuda.empty_cache()

MODEL_NAME   = "google/rembert"
MAX_LEN      = 512
BATCH_SIZE   = 16
ACCUM_STEPS  = 1
EPOCHS       = 5
LR           = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
DROPOUT      = 0.3
PATIENCE     = 3

class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.encodings = tokenizer(
            list(texts),
            max_length=MAX_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        self.labels = torch.tensor(list(labels), dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids":      self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels":         self.labels[idx]
        }

class RemBERTClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(MODEL_NAME)
        self.encoder.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={"use_reentrant": False}
        )
        hidden_size  = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(DROPOUT)
        self.fc      = nn.Linear(hidden_size, 2)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_out = outputs.last_hidden_state[:, 0, :]
        return self.fc(self.dropout(cls_out))

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_dataset = NewsDataset(Xt_Train["clean_text"], y_train.values, tokenizer)
val_dataset   = NewsDataset(Xt_Val["clean_text"],   y_val.values,   tokenizer)
test_dataset  = NewsDataset(Xt_Test["clean_text"],  y_test.values,  tokenizer)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE // ACCUM_STEPS, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE // ACCUM_STEPS)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE // ACCUM_STEPS)

scaler    = torch.amp.GradScaler("cuda")
model     = RemBERTClassifier().to(DEVICE)
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
criterion = nn.CrossEntropyLoss()

total_steps  = (len(train_loader) // ACCUM_STEPS) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler    = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

def run_epoch(loader, training=True):
    model.train() if training else model.eval()
    total_loss, all_preds, all_labels, all_probs = 0, [], [], []

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx, torch.amp.autocast("cuda"):  # autocast wraps the whole epoch, not each batch
        for i, batch in enumerate(loader):
            input_ids = batch["input_ids"].to(DEVICE)
            attn_mask = batch["attention_mask"].to(DEVICE)
            labels    = batch["labels"].to(DEVICE)

            if training:
                logits = model(input_ids, attn_mask)
                loss   = criterion(logits, labels) / ACCUM_STEPS
                scaler.scale(loss).backward()

                if (i + 1) % ACCUM_STEPS == 0 or (i + 1) == len(loader):
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad()
                    scheduler.step()

                loss = loss * ACCUM_STEPS
            else:
                logits = model(input_ids, attn_mask)
                loss   = criterion(logits, labels)

            probs = torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()
            preds = logits.argmax(dim=1).detach().cpu().numpy()
            total_loss += loss.item()
            all_preds  += list(preds)
            all_labels += labels.cpu().numpy().tolist()
            all_probs  += list(probs)

    return total_loss / len(loader), all_preds, all_labels, all_probs

best_val_loss = float("inf")
no_improve    = 0

for epoch in range(1, EPOCHS + 1):
    tr_loss, _, _, _        = run_epoch(train_loader, training=True)
    val_loss, vp, vl, vprob = run_epoch(val_loader,   training=False)

    val_acc = accuracy_score(vl, vp)
    print(f"Epoch {epoch:2d} | train loss {tr_loss:.4f} | "
          f"val loss {val_loss:.4f} | val acc {val_acc:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        no_improve    = 0
        torch.save(model.state_dict(), "rembert_best.pt")
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

model.load_state_dict(torch.load("rembert_best.pt", map_location=DEVICE))
_, tp, tl, tprob = run_epoch(test_loader, training=False)

print_results("RemBERT", accuracy_score(tl, tp), precision_score(tl, tp, zero_division=0), recall_score(tl, tp, zero_division=0), f1_score(tl, tp, zero_division=0), roc_auc_score(tl, tprob))

np.save("rembert_test_probs.npy", tprob)

for _var in ['model', 'train_loader', 'val_loader', 'test_loader',
             'train_dataset', 'val_dataset', 'test_dataset']:
    if _var in globals():
        del globals()[_var]
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
print(f"VRAM free: {torch.cuda.mem_get_info()[0] / 1024**3:.2f} GiB")

NameError: name 'gc' is not defined

In [ ]:
# text model (IndoBERT)
gc.collect()
torch.cuda.empty_cache()

MODEL_NAME = "indobenchmark/indobert-base-p1"

MAX_LENGTH = 512
BATCH_SIZE = 16
EPOCHS = 5
PATIENCE = 3
LEARNING_RATE = 2e-5
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class IndoBERTDataset(Dataset):

    def __init__(self, dataframe, labels, tokenizer, max_length=512):

        self.texts = dataframe["clean_text"].tolist()

        self.labels = labels.reset_index(drop=True).tolist()

        self.tokenizer = tokenizer

        self.max_length = max_length

    def __len__(self):

        return len(self.texts)

    def __getitem__(self, idx):

        text = str(self.texts[idx])

        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label": torch.tensor(label, dtype=torch.long)
        }

train_dataset = IndoBERTDataset(
    Xt_Train,
    y_train,
    tokenizer,
    MAX_LENGTH
)

val_dataset = IndoBERTDataset(
    Xt_Val,
    y_val,
    tokenizer,
    MAX_LENGTH
)

test_dataset = IndoBERTDataset(
    Xt_Test,
    y_test,
    tokenizer,
    MAX_LENGTH
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

class IndoBERTClassifier(nn.Module):

    def __init__(self, dropout=0.3):

        super().__init__()

        self.encoder = AutoModel.from_pretrained(MODEL_NAME)

        hidden_size = self.encoder.config.hidden_size

        self.dropout = nn.Dropout(dropout)

        self.classifier = nn.Linear(hidden_size, 2)

    def forward(self, input_ids, attention_mask):

        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls_output = outputs.last_hidden_state[:, 0, :]

        logits = self.classifier(
            self.dropout(cls_output)
        )

        return logits

def evaluate_model(model, loader, criterion, device):

    model.eval()

    total_loss = 0.0

    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():

        for batch in loader:

            input_ids = batch["input_ids"].to(device)

            attention_mask = batch["attention_mask"].to(device)

            labels = batch["label"].to(device)

            with autocast('cuda'):

                logits = model(
                    input_ids,
                    attention_mask
                )

                loss = criterion(logits, labels)

            total_loss += loss.item()

            probs = F.softmax(
                logits.float(),
                dim=1
            )[:, 1]

            preds = torch.argmax(logits, dim=1)

            all_probs.extend(
                probs.cpu().numpy()
            )

            all_preds.extend(
                preds.cpu().numpy()
            )

            all_labels.extend(
                labels.cpu().numpy()
            )

    avg_loss = total_loss / len(loader)

    acc = accuracy_score(
        all_labels,
        all_preds
    )

    precision = precision_score(
        all_labels,
        all_preds,
        zero_division=0
    )

    recall = recall_score(
        all_labels,
        all_preds,
        zero_division=0
    )

    f1 = f1_score(
        all_labels,
        all_preds,
        zero_division=0
    )

    auc = roc_auc_score(
        all_labels,
        all_probs
    )

    return (
        avg_loss,
        acc,
        precision,
        recall,
        f1,
        auc,
        all_probs,
        all_preds,
        all_labels
    )

def train_model(
    model,
    train_loader,
    val_loader,
    epochs=EPOCHS,
    patience=PATIENCE,
    device=DEVICE
):

    criterion = nn.CrossEntropyLoss()

    optimizer = AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=0.01
    )

    total_steps = len(train_loader) * epochs

    warmup_steps = int(0.10 * total_steps)

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    scaler = GradScaler('cuda')

    best_val_loss = float("inf")

    patience_count = 0

    best_state = None

    for epoch in range(1, epochs + 1):

        model.train()

        total_train_loss = 0.0

        for batch in train_loader:

            input_ids = batch["input_ids"].to(device)

            attention_mask = batch["attention_mask"].to(device)

            labels = batch["label"].to(device)

            optimizer.zero_grad()

            with autocast('cuda'):


                logits = model(
                    input_ids,
                    attention_mask
                )

                loss = criterion(
                    logits,
                    labels
                )

            scaler.scale(loss).backward()

            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                1.0
            )

            scaler.step(optimizer)

            scaler.update()

            scheduler.step()

            total_train_loss += loss.item()

        (
            val_loss,
            val_acc,
            val_prec,
            val_rec,
            val_f1,
            val_auc,
            _,
            _,
            _
        ) = evaluate_model(
            model,
            val_loader,
            criterion,
            device
        )

        print(
            f"Epoch {epoch:02d} | "
            f"Train Loss: {total_train_loss/len(train_loader):.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_acc:.4f} | "
            f"Val F1: {val_f1:.4f} | "
            f"Val AUC: {val_auc:.4f}"
        )

        if val_loss < best_val_loss:

          best_val_loss = val_loss

          patience_count = 0

          best_state = {
              k: v.cpu().clone()
              for k, v in model.state_dict().items()
          }

        else:

          patience_count += 1

          if patience_count >= patience:

              print(f"\nEarly stopping at epoch {epoch}")

              break

    model.load_state_dict(best_state)

    return model

model = IndoBERTClassifier(
    dropout=0.3
).to(DEVICE)

model = train_model(
    model,
    train_loader,
    val_loader
)

criterion = nn.CrossEntropyLoss()

(
    test_loss,
    test_acc,
    test_prec,
    test_rec,
    test_f1,
    test_auc,
    indobert_test_probs,
    indobert_test_preds,
    indobert_test_labels
) = evaluate_model(
    model,
    test_loader,
    criterion,
    DEVICE
)

print_results("IndoBERT", test_acc, test_prec, test_rec, test_f1, test_auc)

np.save("indobert_test_probs.npy", indobert_test_probs)
np.save("y_test.npy", np.array(y_test.reset_index(drop=True).tolist()))

# Cleanup
for _var in ['model', 'train_loader', 'val_loader', 'test_loader',
             'train_dataset', 'val_dataset', 'test_dataset']:
    if _var in globals():
        del globals()[_var]
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
print(f"VRAM free: {torch.cuda.mem_get_info()[0] / 1024**3:.2f} GiB")

In [ ]:
# text model (XLM-RoBERTa)
gc.collect()
torch.cuda.empty_cache()

class TextDataset(Dataset):
    def __init__(self, dataframe, labels, tokenizer, max_length=512):
        self.texts      = dataframe["clean_text"].tolist()
        self.labels     = labels.reset_index(drop=True).tolist()
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label":          torch.tensor(self.labels[idx], dtype=torch.long),
        }

class XLMRobertaClassifier(nn.Module):
    def __init__(self, dropout=0.3):
        super().__init__()
        self.encoder    = XLMRobertaModel.from_pretrained("xlm-roberta-base")
        self.encoder.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={"use_reentrant": False}
        )
        hidden_size     = self.encoder.config.hidden_size
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, 2)

    def forward(self, input_ids, attention_mask):
        outputs    = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        return self.classifier(self.dropout(cls_output))

def evaluate_xlmr(model, loader, criterion, device):
    model.eval()
    total_loss, all_preds, all_labels, all_probs = 0.0, [], [], []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["label"].to(device)
            with autocast('cuda'):
                logits = model(input_ids, attention_mask)
                loss   = criterion(logits, labels)
            total_loss += loss.item()
            probs  = torch.softmax(logits.float(), dim=1)[:, 1]
            preds  = torch.argmax(logits, dim=1)
            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    avg_loss  = total_loss / len(loader)
    acc       = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall    = recall_score(all_labels, all_preds, zero_division=0)
    f1        = f1_score(all_labels, all_preds, zero_division=0)
    auc       = roc_auc_score(all_labels, all_probs)
    return avg_loss, acc, precision, recall, f1, auc, all_probs

def train_xlmr(model, train_loader, val_loader, epochs=5, patience=3, device=DEVICE):
    criterion    = nn.CrossEntropyLoss()
    optimizer    = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
    total_steps  = len(train_loader) * epochs
    warmup_steps = int(0.10 * total_steps)
    scheduler    = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
    )
    scaler = GradScaler()
    best_val_loss, patience_count, best_state = float("inf"), 0, None

    for epoch in range(1, epochs + 1):
        model.train()
        total_train_loss = 0.0
        for batch in train_loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["label"].to(device)
            optimizer.zero_grad()
            with autocast('cuda'):
                logits = model(input_ids, attention_mask)
                loss   = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            total_train_loss += loss.item()

        val_loss, val_acc, _, _, val_f1, val_auc, _ = evaluate_xlmr(
            model, val_loader, criterion, device
        )
        print(f"Epoch {epoch:02d} | Train Loss: {total_train_loss/len(train_loader):.4f} | "
              f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | "
              f"Val F1: {val_f1:.4f} | Val AUC: {val_auc:.4f}")

        if val_loss < best_val_loss:
            best_val_loss, patience_count = val_loss, 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_count += 1
            if patience_count >= patience:
                print(f"Early stopping at epoch {epoch}.")
                break

    model.load_state_dict(best_state)
    return model

xlmr_tokenizer    = XLMRobertaTokenizer.from_pretrained("xlm-roberta-base")
train_xlmr_loader = DataLoader(TextDataset(Xt_Train, y_train, xlmr_tokenizer), batch_size=16, shuffle=True)
val_xlmr_loader   = DataLoader(TextDataset(Xt_Val,   y_val,   xlmr_tokenizer), batch_size=16, shuffle=False)
test_xlmr_loader  = DataLoader(TextDataset(Xt_Test,  y_test,  xlmr_tokenizer), batch_size=16, shuffle=False)

xlmr_model = XLMRobertaClassifier(dropout=0.3).to(DEVICE)
xlmr_model = train_xlmr(xlmr_model, train_xlmr_loader, val_xlmr_loader)

criterion = nn.CrossEntropyLoss()
_, test_acc, test_prec, test_rec, test_f1, test_auc, xlmr_test_probs = evaluate_xlmr(
    xlmr_model, test_xlmr_loader, criterion, DEVICE
)

print_results("XLM-RoBERTa", test_acc, test_prec, test_rec, test_f1, test_auc)

np.save("xlmr_test_probs.npy", xlmr_test_probs)

for _var in ['xlmr_model', 'train_xlmr_loader', 'val_xlmr_loader', 'test_xlmr_loader']:
    if _var in globals():
        del globals()[_var]
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
print(f"VRAM free: {torch.cuda.mem_get_info()[0] / 1024**3:.2f} GiB")

In [ ]:
#Preprocesisng image (CLIP)
clip_processor = CLIPProcessor.from_pretrained(
    "openai/clip-vit-base-patch32"
)

train_image_df_clip = Xi_Train.copy()
train_image_df_clip["label"] = y_train.values

val_image_df_clip = Xi_Val.copy()
val_image_df_clip["label"] = y_val.values

test_image_df_clip = Xi_Test.copy()
test_image_df_clip["label"] = y_test.values

class CLIPImageDataset(Dataset):

    def __init__(self, dataframe, processor):

        self.dataframe = dataframe.reset_index(drop=True)

        self.processor = processor

    def __len__(self):

        return len(self.dataframe)

    def __getitem__(self, idx):

        image_path = self.dataframe.loc[idx, "full_image_path"]

        label = self.dataframe.loc[idx, "label"]

        try:

            image = Image.open(image_path).convert("RGB")

        except:

            # fallback image if corrupted
            image = Image.new("RGB", (224, 224))

        processed = self.processor(
            images=image,
            return_tensors="pt"
        )

        pixel_values = processed["pixel_values"].squeeze(0)

        return {
            "pixel_values": pixel_values,
            "label": torch.tensor(label, dtype=torch.long)
        }

train_clip_dataset = CLIPImageDataset(
    train_image_df_clip,
    clip_processor
)

val_clip_dataset = CLIPImageDataset(
    val_image_df_clip,
    clip_processor
)

test_clip_dataset = CLIPImageDataset(
    test_image_df_clip,
    clip_processor
)

train_clip_loader = DataLoader(
    train_clip_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_clip_loader = DataLoader(
    val_clip_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_clip_loader = DataLoader(
    test_clip_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

batch = next(iter(train_clip_loader))
print("pixel_values shape:", batch["pixel_values"].shape)
print("labels shape:", batch["label"].shape)
batch = next(iter(train_clip_loader))
print("pixel_values shape:", batch["pixel_values"].shape)
print("labels shape:", batch["label"].shape)

In [ ]:
#Preprocesisng image (ResNet & EfficientNet)
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),

    transforms.RandomHorizontalFlip(p=0.3),

    transforms.ColorJitter(
        brightness=0.1,
        contrast=0.1,
        saturation=0.1
    ),

    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_image_df = Xi_Train.copy()
train_image_df["label"] = y_train.values

val_image_df = Xi_Val.copy()
val_image_df["label"] = y_val.values

test_image_df = Xi_Test.copy()
test_image_df["label"] = y_test.values

class NewsImageDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):

        image_path = self.dataframe.loc[idx, "full_image_path"]
        label = self.dataframe.loc[idx, "label"]

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

train_image_dataset = NewsImageDataset(
    train_image_df,
    transform=train_transform
)

val_image_dataset = NewsImageDataset(
    val_image_df,
    transform=test_transform
)

test_image_dataset = NewsImageDataset(
    test_image_df,
    transform=test_transform
)

train_image_loader = DataLoader(
    train_image_dataset,
    batch_size=16,
    shuffle=True
)

val_image_loader = DataLoader(
    val_image_dataset,
    batch_size=16,
    shuffle=False
)

test_image_loader = DataLoader(
    test_image_dataset,
    batch_size=16,
    shuffle=False
)

images, labels = next(iter(train_image_loader))

print(images.shape)
print(labels.shape)

In [ ]:
# Image model (ResNet-50)
gc.collect()
torch.cuda.empty_cache()

class ResNetClassifier(nn.Module):
    def __init__(self, dropout=0.3):
        super().__init__()
        self.backbone        = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        in_features          = self.backbone.fc.in_features
        self.backbone.fc     = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_features, 2)
        )

    def forward(self, images):
        return self.backbone(images)

def evaluate_resnet(model, loader, criterion, device):
    model.eval()
    total_loss, all_preds, all_labels, all_probs = 0.0, [], [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            with torch.amp.autocast('cuda'):
                logits = model(images)
                loss   = criterion(logits, labels)
            total_loss += loss.item()
            probs  = torch.softmax(logits.float(), dim=1)[:, 1]
            preds  = torch.argmax(logits, dim=1)
            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    avg_loss  = total_loss / len(loader)
    acc       = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall    = recall_score(all_labels, all_preds, zero_division=0)
    f1        = f1_score(all_labels, all_preds, zero_division=0)
    auc       = roc_auc_score(all_labels, all_probs)
    return avg_loss, acc, precision, recall, f1, auc, all_probs, all_labels


def train_resnet(model, train_loader, val_loader, epochs=5, patience=3, device=DEVICE):
    criterion  = nn.CrossEntropyLoss()
    optimizer  = AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
    scaler = torch.amp.GradScaler('cuda')

    total_steps  = len(train_loader) * epochs
    warmup_steps = int(0.10 * total_steps)
    scheduler    = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
    )

    best_val_loss, patience_count, best_state = float("inf"), 0, None

    for epoch in range(1, epochs + 1):
        model.train()
        total_train_loss = 0.0
        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()
            with torch.amp.autocast('cuda'):
                logits = model(images)
                loss   = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            total_train_loss += loss.item()

        val_loss, val_acc, _, _, val_f1, val_auc, _, _ = evaluate_resnet(
        model, val_loader, criterion, device)

        print(f"Epoch {epoch:02d} | Train Loss: {total_train_loss/len(train_loader):.4f} | "
              f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | "
              f"Val F1: {val_f1:.4f} | Val AUC: {val_auc:.4f}")

        if val_loss < best_val_loss:
            best_val_loss, patience_count = val_loss, 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_count += 1
            if patience_count >= patience:
                print(f"Early stopping at epoch {epoch}.")
                break

    model.load_state_dict(best_state)
    return model

resnet_model = ResNetClassifier(dropout=0.3).to(DEVICE)
resnet_model = train_resnet(resnet_model, train_image_loader, val_image_loader)

criterion = nn.CrossEntropyLoss()
_, test_acc, test_prec, test_rec, test_f1, test_auc, resnet_test_probs, resnet_test_labels = evaluate_resnet(
    resnet_model, test_image_loader, criterion, DEVICE
)

print_results("ResNet-50", test_acc, test_prec, test_rec, test_f1, test_auc)

np.save("resnet_test_probs.npy", resnet_test_probs)

for _var in ['resnet_model', 'train_image_loader', 'val_image_loader', 'test_image_loader',
             'train_image_dataset', 'val_image_dataset', 'test_image_dataset',
             'train_image_df', 'val_image_df', 'test_image_df']:
    if _var in globals():
        del globals()[_var]
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
print(f"VRAM free: {torch.cuda.mem_get_info()[0] / 1024**3:.2f} GiB")

In [ ]:
# Image model (EfficientNet)
gc.collect()
torch.cuda.empty_cache()

class EfficientNetClassifier(nn.Module):
    def __init__(self, dropout=0.3):
        super().__init__()
        self.backbone   = timm.create_model("efficientnet_b3", pretrained=True, num_classes=0)
        num_features    = self.backbone.num_features  # 1536
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(num_features, 2)

    def forward(self, images):
        features = self.backbone(images)
        return self.classifier(self.dropout(features))

def evaluate_efficientnet(model, loader, criterion, device):
    model.eval()
    total_loss, all_preds, all_labels, all_probs = 0.0, [], [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            with autocast('cuda'):
                logits = model(images)
                loss   = criterion(logits, labels)
            total_loss += loss.item()
            probs  = torch.softmax(logits.float(), dim=1)[:, 1]
            preds  = torch.argmax(logits, dim=1)
            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    avg_loss  = total_loss / len(loader)
    acc       = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall    = recall_score(all_labels, all_preds, zero_division=0)
    f1        = f1_score(all_labels, all_preds, zero_division=0)
    auc       = roc_auc_score(all_labels, all_probs)
    return avg_loss, acc, precision, recall, f1, auc, all_probs

def train_efficientnet(model, train_loader, val_loader, epochs=5, patience=3, device=DEVICE):
    criterion  = nn.CrossEntropyLoss()
    optimizer  = AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
    scaler     = GradScaler('cuda')
    total_steps  = len(train_loader) * epochs
    warmup_steps = int(0.10 * total_steps)
    scheduler    = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
    )

    best_val_loss, patience_count, best_state = float("inf"), 0, None

    for epoch in range(1, epochs + 1):
        model.train()
        total_train_loss = 0.0
        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()
            with autocast('cuda'):
                logits = model(images)
                loss   = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            total_train_loss += loss.item()

        val_loss, val_acc, _, _, val_f1, val_auc, _ = evaluate_efficientnet(
            model, val_loader, criterion, device
        )
        print(f"Epoch {epoch:02d} | Train Loss: {total_train_loss/len(train_loader):.4f} | "
              f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | "
              f"Val F1: {val_f1:.4f} | Val AUC: {val_auc:.4f}")

        if val_loss < best_val_loss:
            best_val_loss, patience_count = val_loss, 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_count += 1
            if patience_count >= patience:
                print(f"Early stopping at epoch {epoch}.")
                break

    model.load_state_dict(best_state)
    return model

efficientnet_model = EfficientNetClassifier(dropout=0.3).to(DEVICE)
efficientnet_model = train_efficientnet(efficientnet_model, train_image_loader, val_image_loader)

criterion = nn.CrossEntropyLoss()
_, test_acc, test_prec, test_rec, test_f1, test_auc, effnet_test_probs = evaluate_efficientnet(
    efficientnet_model, test_image_loader, criterion, DEVICE
)

print_results("EfficientNet-B3", test_acc, test_prec, test_rec, test_f1, test_auc)

np.save("effnet_test_probs.npy", effnet_test_probs)

for _var in ['efficientnet_model', 'train_image_loader', 'val_image_loader', 'test_image_loader',
             'train_image_dataset', 'val_image_dataset', 'test_image_dataset',
             'train_image_df', 'val_image_df', 'test_image_df']:
    if _var in globals():
        del globals()[_var]
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
print(f"VRAM free: {torch.cuda.mem_get_info()[0] / 1024**3:.2f} GiB")

In [ ]:
# Image model (CLIP)
gc.collect()
torch.cuda.empty_cache()

MODEL_NAME = "openai/clip-vit-base-patch32"

BATCH_SIZE = 16
EPOCHS = 5
PATIENCE = 3
LEARNING_RATE = 1e-4

class CLIPClassifier(nn.Module):

    def __init__(self, dropout=0.3):

        super().__init__()

        self.clip = CLIPModel.from_pretrained(
            MODEL_NAME
        )


        hidden_size = self.clip.vision_model.config.hidden_size

        self.dropout = nn.Dropout(dropout)

        self.classifier = nn.Linear(hidden_size, 2)

    def forward(self, pixel_values):

      outputs = self.clip.vision_model(
          pixel_values=pixel_values
      )

      pooled_output = outputs.pooler_output

      logits = self.classifier(
          self.dropout(pooled_output)
      )

      return logits

def evaluate_clip(model, loader, criterion, device):

    model.eval()

    total_loss = 0.0

    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():

        for batch in loader:

            pixel_values = batch["pixel_values"].to(device)

            labels = batch["label"].to(device)

            with autocast('cuda'):

                logits = model(pixel_values)

                loss = criterion(logits, labels)

            total_loss += loss.item()

            probs = F.softmax(
                logits.float(),
                dim=1
            )[:, 1]

            preds = torch.argmax(
                logits,
                dim=1
            )

            all_probs.extend(
                probs.cpu().numpy()
            )

            all_preds.extend(
                preds.cpu().numpy()
            )

            all_labels.extend(
                labels.cpu().numpy()
            )

    avg_loss = total_loss / len(loader)

    acc = accuracy_score(
        all_labels,
        all_preds
    )

    precision = precision_score(
        all_labels,
        all_preds,
        zero_division=0
    )

    recall = recall_score(
        all_labels,
        all_preds,
        zero_division=0
    )

    f1 = f1_score(
        all_labels,
        all_preds,
        zero_division=0
    )

    auc = roc_auc_score(
        all_labels,
        all_probs
    )

    return (
        avg_loss,
        acc,
        precision,
        recall,
        f1,
        auc,
        all_probs,
        all_preds,
        all_labels
    )

def train_clip(
    model,
    train_loader,
    val_loader,
    epochs=EPOCHS,
    patience=PATIENCE,
    device=DEVICE
):

    criterion = nn.CrossEntropyLoss()

    optimizer = AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=0.01
    )

    total_steps = len(train_loader) * epochs

    warmup_steps = int(0.10 * total_steps)

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    scaler = GradScaler('cuda')

    best_val_loss = float("inf")

    patience_count = 0

    best_state = None

    for epoch in range(1, epochs + 1):

        model.train()

        total_train_loss = 0.0

        for batch in train_loader:

            pixel_values = batch["pixel_values"].to(device)

            labels = batch["label"].to(device)

            optimizer.zero_grad()

            with autocast('cuda'):

                logits = model(pixel_values)

                loss = criterion(
                    logits,
                    labels
                )

            scaler.scale(loss).backward()

            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                1.0
            )

            scaler.step(optimizer)

            scaler.update()

            scheduler.step()

            total_train_loss += loss.item()

        (
            val_loss,
            val_acc,
            val_prec,
            val_rec,
            val_f1,
            val_auc,
            _,
            _,
            _
        ) = evaluate_clip(
            model,
            val_loader,
            criterion,
            device
        )

        print(
            f"Epoch {epoch:02d} | "
            f"Train Loss: {total_train_loss/len(train_loader):.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_acc:.4f} | "
            f"Val F1: {val_f1:.4f} | "
            f"Val AUC: {val_auc:.4f}"
        )

        if val_loss < best_val_loss:

            best_val_loss = val_loss

            patience_count = 0

            best_state = {
                k: v.cpu().clone()
                for k, v in model.state_dict().items()
            }

        else:

            patience_count += 1

            if patience_count >= patience:

                print(f"\nEarly stopping at epoch {epoch}")

                break

    model.load_state_dict(best_state)

    return model

clip_model = CLIPClassifier(
    dropout=0.3
).to(DEVICE)

clip_model = train_clip(
    clip_model,
    train_clip_loader,
    val_clip_loader
)

criterion = nn.CrossEntropyLoss()

(
    test_loss,
    test_acc,
    test_prec,
    test_rec,
    test_f1,
    test_auc,
    clip_test_probs,
    clip_test_preds,
    clip_test_labels
) = evaluate_clip(
    clip_model,
    test_clip_loader,
    criterion,
    DEVICE
)

print_results("CLIP", test_acc, test_prec, test_rec, test_f1, test_auc)

np.save("clip_test_probs.npy", clip_test_probs)

for _var in ['clip_model', 'train_clip_loader', 'val_clip_loader', 'test_clip_loader',
             'train_clip_dataset', 'val_clip_dataset', 'test_clip_dataset',
             'train_image_df_clip', 'val_image_df_clip', 'test_image_df_clip']:
    if _var in globals():
        del globals()[_var]
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
print(f"VRAM free: {torch.cuda.mem_get_info()[0] / 1024**3:.2f} GiB")


In [ ]:
# ── Load all saved probs ───────────────────────────────────────────────────
indobert_test_probs = np.load("indobert_test_probs.npy")
xlmr_test_probs     = np.load("xlmr_test_probs.npy")
rembert_test_probs  = np.load("rembert_test_probs.npy")
resnet_test_probs   = np.load("resnet_test_probs.npy")
effnet_test_probs   = np.load("effnet_test_probs.npy")
clip_test_probs     = np.load("clip_test_probs.npy")
true_labels         = np.load("y_test.npy").tolist()

# ── Run all 9 pairings ─────────────────────────────────────────────────────
run_late_fusion("IndoBERT + CLIP",              indobert_test_probs, clip_test_probs,   true_labels)
run_late_fusion("IndoBERT + EfficientNet",      indobert_test_probs, effnet_test_probs, true_labels)
run_late_fusion("IndoBERT + ResNet-50",         indobert_test_probs, resnet_test_probs, true_labels)
run_late_fusion("XLM-RoBERTa + CLIP",           xlmr_test_probs,     clip_test_probs,   true_labels)
run_late_fusion("XLM-RoBERTa + EfficientNet",   xlmr_test_probs,     effnet_test_probs, true_labels)
run_late_fusion("XLM-RoBERTa + ResNet-50",      xlmr_test_probs,     resnet_test_probs, true_labels)
run_late_fusion("RemBERT + CLIP",               rembert_test_probs,  clip_test_probs,   true_labels)
run_late_fusion("RemBERT + EfficientNet",        rembert_test_probs,  effnet_test_probs, true_labels)
run_late_fusion("RemBERT + ResNet-50",           rembert_test_probs,  resnet_test_probs, true_labels)